# Fase 5 MVP - Preparación de datos auditable

Este notebook permite auditar la preparación de datos del MVP end-to-end sin sobrescribir los artefactos ya generados en `FASE5/outputs`.

Contrato operativo:

- No llama a `build_phase5(save=True)`.
- No escribe CSV, Excel, JSON ni manifiestos.
- Recalcula la Fase 5 solo en memoria con `save=False`.
- Verifica al final que los hashes de `FASE5/outputs` no hayan cambiado durante la ejecución.

In [1]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    MVP_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "FASE5":
    MVP_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "F5_F8_MVP":
    MVP_ROOT = PROJECT_ROOT
else:
    MVP_ROOT = Path("/home/pablo/Research_LeyIA_DataScience/Research_AI_law/F5_F8_MVP")

if str(MVP_ROOT) not in sys.path:
    sys.path.insert(0, str(MVP_ROOT))

from FASE5.src.build import build_phase5
from FASE5.src.transform import get_log_variables
from FASE5.src.variables import get_mvp_variables

OUTPUTS = MVP_ROOT / "FASE5" / "outputs"
NOTEBOOK_PATH = MVP_ROOT / "FASE5" / "notebooks" / "05_data_preparation.ipynb"

assert OUTPUTS.exists(), OUTPUTS
print(f"MVP_ROOT: {MVP_ROOT}")
print(f"OUTPUTS: {OUTPUTS}")

MVP_ROOT: /home/pablo/Research_LeyIA_DataScience/Research_AI_law/F5_F8_MVP
OUTPUTS: /home/pablo/Research_LeyIA_DataScience/Research_AI_law/F5_F8_MVP/FASE5/outputs


In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

tracked_outputs = sorted(
    p for p in OUTPUTS.iterdir()
    if p.is_file() and p.suffix.lower() in {".csv", ".xlsx", ".json"}
)
initial_hashes = {p.name: sha256_file(p) for p in tracked_outputs}
initial_hashes

{'MVP_AUDITABLE.xlsx': '64ce2eed15d76130e4da752222f36a753a672b015824b4d438320825fa8c71bc',
 'coverage_report_mvp.csv': '40a6f1b2121ef694d47a1acd5e2e9a1fefd9f0432d043b11cd0711331644ba97',
 'fase5_manifest.json': 'a36de1bb6ea9ab30c192f9006739352a13acfd0b284e15e37b6b29bda28b17fb',
 'feature_matrix_mvp.csv': 'a8744becf7dc188289c864f397c4dff5b5651e3c1be4c81d000a352a0b81585d',
 'mvp_countries.csv': '733a39f0ce0ae48800f19e973bf0792ff60bd11f3cf3fea88365d920c4082354',
 'mvp_train_test_split.csv': '6c270efcfcadb79735b313da0ad323d38ec2f083b3e0ff912edde1387b1d3ca1',
 'mvp_transform_params.csv': '30b80bffb198387731f0613a228f4aee9be32cf049bd91a1f34be592782ed68a',
 'mvp_variables_catalog.csv': '981fc0516b8aab29bc58aa7368e8cbc261ac4c9922b7e791b66a8183b0498529'}

## 1. Carga de artefactos oficiales

Esta sección carga solamente archivos ya existentes de `FASE5/outputs` para revisar dimensiones, cobertura y manifiesto.

In [3]:
feature_matrix = pd.read_csv(OUTPUTS / "feature_matrix_mvp.csv")
coverage = pd.read_csv(OUTPUTS / "coverage_report_mvp.csv")
countries = pd.read_csv(OUTPUTS / "mvp_countries.csv")
variables_catalog = pd.read_csv(OUTPUTS / "mvp_variables_catalog.csv")
transform_params = pd.read_csv(OUTPUTS / "mvp_transform_params.csv")
split = pd.read_csv(OUTPUTS / "mvp_train_test_split.csv")
manifest = json.loads((OUTPUTS / "fase5_manifest.json").read_text(encoding="utf-8"))

summary = pd.DataFrame([
    {"artefacto": "feature_matrix_mvp.csv", "filas": feature_matrix.shape[0], "columnas": feature_matrix.shape[1]},
    {"artefacto": "coverage_report_mvp.csv", "filas": coverage.shape[0], "columnas": coverage.shape[1]},
    {"artefacto": "mvp_countries.csv", "filas": countries.shape[0], "columnas": countries.shape[1]},
    {"artefacto": "mvp_variables_catalog.csv", "filas": variables_catalog.shape[0], "columnas": variables_catalog.shape[1]},
    {"artefacto": "mvp_transform_params.csv", "filas": transform_params.shape[0], "columnas": transform_params.shape[1]},
    {"artefacto": "mvp_train_test_split.csv", "filas": split.shape[0], "columnas": split.shape[1]},
])
summary

,artefacto,filas,columnas
0,feature_matrix_mvp.csv,43,126
1,coverage_report_mvp.csv,40,6
2,mvp_countries.csv,43,7
3,mvp_variables_catalog.csv,40,17
4,mvp_transform_params.csv,56,8
5,mvp_train_test_split.csv,43,4


In [4]:
assert feature_matrix.shape[0] == 43
assert countries.shape[0] == 43
assert coverage.shape[0] == 40
assert variables_catalog.shape[0] == 40
assert feature_matrix["iso3"].is_unique
assert "CHL" in set(feature_matrix["iso3"])
assert manifest["rules"]["no_imputation"] is True
assert manifest["rules"]["phase3_phase4_immutable"] is True
print("Contratos principales OK")

Contratos principales OK


## 2. Muestra MVP

La muestra de 43 países combina Top 30 Microsoft disponibles, Chile, países obligatorios y peers regionales/prioritarios.

In [5]:
countries[["iso3", "country_name_canonical", "region", "income_group", "category", "rank_ms", "reason"]].sort_values(["category", "rank_ms", "iso3"]).head(50)

,iso3,country_name_canonical,region,income_group,category,rank_ms,reason
40,GRC,Greece,Europe & Central Asia,High income,eu_laggard,NaN,EU sur con bajo desarrollo IA relativo
42,HRV,Croatia,Europe & Central Asia,High income,eu_laggard,NaN,"EU pequeno, contraste regulatorio"
41,ROU,Romania,Europe & Central Asia,High income,eu_laggard,NaN,"EU este, contraste con lideres"
28,CHL,Chile,Latin America & Caribbean,High income,focal,NaN,Caso focal; Boletin 16821-19 Ley Marco IA
34,ARG,Argentina,Latin America & Caribbean,Upper middle income,latam_peer,NaN,Peer regional de Chile
35,BRA,Brazil,Latin America & Caribbean,Upper middle income,latam_peer,NaN,"Mayor economia LATAM, marco IA federal"
36,COL,Colombia,Latin America & Caribbean,Upper middle income,latam_peer,NaN,Peer economico de Chile
37,MEX,Mexico,Latin America & Caribbean,Upper middle income,latam_peer,NaN,OECD + LATAM
38,PER,Peru,Latin America & Caribbean,Upper middle income,latam_peer,NaN,Peer regional
39,URY,Uruguay,Latin America & Caribbean,High income,latam_peer,NaN,Gobierno digital LATAM


In [6]:
countries.groupby(["category"], dropna=False)["iso3"].count().rename("n_paises").reset_index().sort_values("n_paises", ascending=False)

,category,n_paises
4,ms_top30,28
2,latam_peer,6
3,mandatory,5
0,eu_laggard,3
1,focal,1


## 3. Variables observadas y cobertura

La matriz usa 40 variables reales observadas, verificadas contra el diccionario de Fase 3. No se crean variables fantasma.

In [7]:
coverage.sort_values("pct_complete").reset_index(drop=True)

,variable_matriz,n_total,n_non_null,n_missing,pct_complete,below_threshold
0,iapp_ley_ia_vigente,43,18,25,41.86,False
1,iapp_categoria_obligatoriedad,43,18,25,41.86,False
2,iapp_proyecto_ley_ia,43,18,25,41.86,False
3,iapp_modelo_gobernanza,43,18,25,41.86,False
4,iapp_n_leyes_relacionadas,43,18,25,41.86,False
5,iapp_n_autoridades,43,18,25,41.86,False
6,stanford_fig_6_3_5_fig_6_3_5_volume_of_publica...,43,26,17,60.47,False
7,oecd_5_ict_business_oecd_biz_bigdata_pct,43,29,14,67.44,False
8,oecd_5_ict_business_oecd_biz_ai_pct,43,31,12,72.09,False
9,wb_researchers_rd_per_million,43,36,7,83.72,False


In [8]:
assert not coverage["below_threshold"].any()
assert coverage["pct_complete"].ge(30.0).all()

fig, ax = plt.subplots(figsize=(8, 10))
plot_df = coverage.sort_values("pct_complete")
ax.barh(plot_df["variable_matriz"], plot_df["pct_complete"])
ax.axvline(30, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Cobertura en muestra MVP (%)")
ax.set_ylabel("")
ax.set_title("Cobertura de variables Fase 5")
plt.tight_layout()
plt.show()

## 4. Rebuild en memoria

Aquí se ejecuta la preparación Fase 5 con `save=False`; esto recalcula los objetos en memoria y evita escribir en disco.

In [9]:
rebuilt = build_phase5(save=False)
rebuilt_feature_matrix = rebuilt["feature_matrix"]
rebuilt_coverage = rebuilt["coverage"]

assert rebuilt_feature_matrix.shape == feature_matrix.shape
assert rebuilt_coverage.shape == coverage.shape
assert set(rebuilt_feature_matrix["iso3"]) == set(feature_matrix["iso3"])
assert list(rebuilt_coverage["variable_matriz"]) == list(coverage["variable_matriz"])
print("Rebuild en memoria OK")

Rebuild en memoria OK


## 5. No imputación

El contrato de Fase 5 preserva valores faltantes. Las transformaciones log y robust z-score mantienen el mismo patrón de missingness que sus variables fuente.

In [10]:
curated = rebuilt["curated"].sort_values("iso3").set_index("iso3")
fm = rebuilt_feature_matrix.sort_values("iso3").set_index("iso3")

missing_checks = []
for variable in get_mvp_variables():
    missing_checks.append({
        "variable": variable,
        "tipo_check": "original",
        "missing_preservado": fm[variable].isna().equals(curated[variable].isna()),
        "n_missing": int(fm[variable].isna().sum()),
    })

for variable in get_log_variables():
    log_col = f"{variable}_log"
    if log_col in fm.columns:
        missing_checks.append({
            "variable": log_col,
            "tipo_check": "log_vs_source",
            "missing_preservado": fm[log_col].isna().equals(fm[variable].isna()),
            "n_missing": int(fm[log_col].isna().sum()),
        })

for source_col in list(fm.columns):
    z_col = f"{source_col}_z"
    if z_col in fm.columns:
        missing_checks.append({
            "variable": z_col,
            "tipo_check": "z_vs_source",
            "missing_preservado": fm[z_col].isna().equals(fm[source_col].isna()),
            "n_missing": int(fm[z_col].isna().sum()),
        })

missing_report = pd.DataFrame(missing_checks)
assert missing_report["missing_preservado"].all()
missing_report.sort_values(["tipo_check", "variable"]).head(80)

,variable,tipo_check,missing_preservado,n_missing
40,oxford_ind_ai_unicorns_log_log,log_vs_source,True,0
41,oxford_ind_non_ai_unicorns_log_log,log_vs_source,True,0
42,oxford_ind_vc_availability_log,log_vs_source,True,0
48,wb_electric_consumption_kwh_pc_log,log_vs_source,True,1
43,wb_fdi_net_inflows_log,log_vs_source,True,1
...,...,...,...,...
90,wb_fdi_net_inflows_log_z,z_vs_source,True,1
58,wb_fdi_net_inflows_z,z_vs_source,True,1
91,wb_gdp_per_capita_ppp_log_z,z_vs_source,True,1
75,wb_gdp_per_capita_ppp_z,z_vs_source,True,1


## 6. Transformaciones y split

Los parámetros de transformaciones quedan fuera de la matriz analítica y se auditan en `mvp_transform_params.csv`.

In [11]:
assert not any(col.endswith("_transform_method") for col in feature_matrix.columns)
assert {"log_transform", "robust_zscore"}.issubset(set(transform_params["transform"]))
transform_params.head(30)

,source_variable,derived_variable,transform,method,center_median,scale_mad,n_non_null,status
0,iapp_ley_ia_vigente,iapp_ley_ia_vigente_z,robust_zscore,median_mad,0.000000e+00,0.000000e+00,18,zero_mad_or_not_estimable
1,iapp_proyecto_ley_ia,iapp_proyecto_ley_ia_z,robust_zscore,median_mad,1.000000e+00,0.000000e+00,18,zero_mad_or_not_estimable
2,iapp_n_leyes_relacionadas,iapp_n_leyes_relacionadas_z,robust_zscore,median_mad,7.500000e+00,1.500000e+00,18,ok
3,iapp_n_autoridades,iapp_n_autoridades_z,robust_zscore,median_mad,7.000000e+00,1.500000e+00,18,ok
4,oxford_ind_company_investment_emerging_tech,oxford_ind_company_investment_emerging_tech_z,robust_zscore,median_mad,6.425000e+01,1.800000e+01,43,ok
5,oxford_ind_ai_unicorns_log,oxford_ind_ai_unicorns_log_log,log_transform,log1p,NaN,NaN,43,ok
6,oxford_ind_ai_unicorns_log,oxford_ind_ai_unicorns_log_z,robust_zscore,median_mad,0.000000e+00,0.000000e+00,43,zero_mad_or_not_estimable
7,oxford_ind_ai_unicorns_log_log,oxford_ind_ai_unicorns_log_log_z,robust_zscore,median_mad,0.000000e+00,0.000000e+00,43,zero_mad_or_not_estimable
8,oxford_ind_non_ai_unicorns_log,oxford_ind_non_ai_unicorns_log_log,log_transform,log1p,NaN,NaN,43,ok
9,oxford_ind_non_ai_unicorns_log,oxford_ind_non_ai_unicorns_log_z,robust_zscore,median_mad,1.816677e+01,1.816677e+01,43,ok


In [12]:
split_summary = split.groupby("split")["iso3"].count().rename("n_paises").reset_index()
assert set(split["split"]) == {"train", "test"}
split_summary

,split,n_paises
0,test,9
1,train,34


## 7. Chile y Excel auditable

Se revisa que Chile esté incluido y que el Excel conserve las hojas esperadas y que separe hipotesis, protocolo de auditoria, matriz humana 43 x 40 y matriz tecnica 43 x 126.

In [13]:
chile = feature_matrix.loc[feature_matrix["iso3"] == "CHL"].T.rename(columns={feature_matrix.loc[feature_matrix["iso3"] == "CHL"].index[0]: "CHL"})
chile.head(60)

,CHL
iso3,CHL
country_name_canonical,Chile
entity_type,country_iso3
region,Latin America & Caribbean
income_group,High income
n_sources_present,8
source_list,"iapp, microsoft, oxford, wb, wipo, stanford, o..."
n_variables_available,388
pct_variables_available,97.73
included_in_matrix,True


In [14]:
from openpyxl import load_workbook

workbook = load_workbook(OUTPUTS / "MVP_AUDITABLE.xlsx", read_only=True)
expected_sheets = [
    "0_Leer_Primero",
    "1_Hipotesis",
    "2_Como_Auditar",
    "3_Paises_43",
    "4_Ingreso_Region",
    "5_Variables_40",
    "6_Matriz_40_Humana",
    "7_Leyenda_Colores",
    "8_Casos_Atencion",
    "9_Normalizacion",
    "10_Cobertura",
    "11_Features_Fase6",
    "12_Diccionario_Cols",
    "13_Trazabilidad",
    "14_Transformaciones",
]
assert workbook.sheetnames == expected_sheets
assert workbook["1_Hipotesis"].max_row == 14
assert workbook["5_Variables_40"].max_row == 41
assert workbook["6_Matriz_40_Humana"].max_row == 44
assert workbook["6_Matriz_40_Humana"].max_column == 44
assert workbook["9_Normalizacion"].max_row >= 12
assert workbook["11_Features_Fase6"].max_row == 44
assert workbook["11_Features_Fase6"].max_column == 126
workbook.sheetnames

['0_Leer_Primero',
 '1_Hipotesis',
 '2_Como_Auditar',
 '3_Paises_43',
 '4_Ingreso_Region',
 '5_Variables_40',
 '6_Matriz_40_Humana',
 '7_Leyenda_Colores',
 '8_Casos_Atencion',
 '9_Normalizacion',
 '10_Cobertura',
 '11_Features_Fase6',
 '12_Diccionario_Cols',
 '13_Trazabilidad',
 '14_Transformaciones']

## 8. Guardia anti-sobrescritura

La última celda compara los hashes iniciales y finales de todos los artefactos oficiales de `FASE5/outputs`. Si algún archivo cambió al ejecutar el notebook, esta celda falla.

In [15]:
final_hashes = {p.name: sha256_file(p) for p in tracked_outputs}
changed = {
    name: {"antes": initial_hashes[name], "despues": final_hashes.get(name)}
    for name in initial_hashes
    if final_hashes.get(name) != initial_hashes[name]
}
assert not changed, changed
print("Notebook audit-only OK: ningun artefacto de FASE5/outputs cambio durante la ejecucion.")

Notebook audit-only OK: ningun artefacto de FASE5/outputs cambio durante la ejecucion.
